### 1.0 - Libraries 

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit.library import qaoa_ansatz
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

### 2.0 - Data 

In [ ]:
# Define your QUBO matrix Q (Goal: minimize x^T @ Q @ x, x in {0, 1}^n)
# Optimal solution for this example is x = [1, 0, 1] with cost = -5.0
Q = np.array([
    [-2.0,  2.0,  0.0],
    [ 2.0, -2.0,  1.0],
    [ 0.0,  1.0, -3.0]
])
Q = 0.5 * (Q + Q.T)

### 3.0 - QUBO to Hamiltonian 

In [ ]:
# Convert QUBO matrix Q to Ising Hamiltonian (SparsePauliOp)
def qubo_to_ising(Q_mat):
    n = Q_mat.shape[0]
    pauli_list = []
    offset = 0.0

    for i in range(n):
        h_i = -0.5 * np.sum(Q_mat[i, :])
        offset += 0.5 * Q_mat[i, i]
        if abs(h_i) > 1e-10:
            label = ["I"] * n
            label[n - 1 - i] = "Z"
            pauli_list.append(("".join(label), h_i))

    for i in range(n):
        for j in range(i + 1, n):
            J_ij = 0.5 * Q_mat[i, j]
            offset += 0.5 * Q_mat[i, j]
            if abs(J_ij) > 1e-10:
                label = ["I"] * n
                label[n - 1 - i] = "Z"
                label[n - 1 - j] = "Z"
                pauli_list.append(("".join(label), J_ij))

    return SparsePauliOp.from_list(pauli_list), offset

cost_hamiltonian, _ = qubo_to_ising(Q)

### 4.0 - Circuit 

In [ ]:
# 3. Build abstract QAOA circuit
reps = 2
ansatz = qaoa_ansatz(cost_operator=cost_hamiltonian, reps=reps)
ansatz.measure_all()

### 5.0 - Functions 

In [ ]:
# Classical QUBO evaluation helper
def evaluate_qubo(bitstring):
    x = np.array([int(b) for b in bitstring[::-1]])
    return float(x @ Q @ x)

In [ ]:
# Optimize parameters LOCALLY (0 QPU time used)
local_sampler = StatevectorSampler(seed=42)

def local_cost_func(params):
    job = local_sampler.run([(ansatz, params)], shots=2048)
    counts = job.result()[0].data.meas.get_counts()
    total_shots = sum(counts.values())
    return sum(evaluate_qubo(bits) * count for bits, count in counts.items()) / total_shots

### 6.0 - Optimize (Local) 

In [ ]:
rng = np.random.default_rng(42)
init_params = rng.uniform(0, np.pi, ansatz.num_parameters)
opt_result = minimize(local_cost_func, init_params, method="COBYLA", options={"maxiter": 200})
optimal_params = opt_result.x
print(f"Local optimization complete. Simulated min expectation: {opt_result.fun:.4f}")

### 7.0 - Run Circuit 

In [ ]:
# Connect to IBM Quantum & select the least busy real QPU
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Target IBM backend: {backend.name}")

In [ ]:
# Transpile circuit to hardware ISA (Mandatory for IBM Runtime)
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_ansatz = pm.run(ansatz)

In [ ]:
# Submit job in Job mode 
hw_sampler = Sampler(mode=backend)
job = hw_sampler.run([(isa_ansatz, optimal_params)], shots=2048)
print(f"Submitted Job ID: {job.job_id()} — waiting for QPU execution...")

### 8.0 - Post Processing 

In [ ]:
# Retrieve and display hardware results
hw_counts = job.result()[0].data.meas.get_counts()
sorted_counts = sorted(hw_counts.items(), key=lambda item: item[1], reverse=True)

In [ ]:
print(f"Minimum expectation value found: {opt_result.fun:.4f}\n")
print("Top sampled solutions [x0, x1, x2]:")
for bitstring, count in sorted_counts[:5]:
    x_vec = [int(b) for b in bitstring[::-1]]
    cost = evaluate_qubo(bitstring)
    print(f"  x = {x_vec} | Shots: {count:4d} | QUBO Cost: {cost:.2f}")

In [ ]:
print(job) 